# EDA bộ dữ liệu ViMD

Notebook phân tích bộ dữ liệu [`nguyendv02/ViMD_Dataset`](https://huggingface.co/datasets/nguyendv02/ViMD_Dataset) — bộ tiếng nói đa phương ngữ Việt Nam từ các đài phát thanh/truyền hình địa phương.

Phạm vi phân tích:

- đọc metadata của toàn bộ shard Parquet mà không tải cột audio;
- kiểm tra quy mô và mức khớp với thống kê công bố;
- phân tích split, vùng, tỉnh, giới tính và người nói;
- kiểm tra mất cân bằng và độ phủ tỉnh giữa train/valid/test;
- phát hiện speaker, filename hoặc transcript bị rò rỉ giữa các split;
- phân tích transcript và tùy chọn kiểm tra tín hiệu trên một số audio mẫu;
- xuất metadata đã làm giàu và shortlist phục vụ nghe thủ công.

> Dataset card ghi giấy phép **CC BY-NC-ND 4.0** và mục đích nghiên cứu. Điều khoản `NC` và `ND` cần được xem xét kỹ trước khi tạo dữ liệu phái sinh, fine-tune hoặc clone giọng. Notebook này chỉ hỗ trợ phân tích dữ liệu, không cấp thêm quyền sử dụng.

In [ ]:
%pip install -q "datasets>=3.0" "huggingface_hub>=0.25" fsspec pyarrow pandas numpy matplotlib seaborn soundfile librosa tqdm

## 1. Cấu hình

Repository có dung lượng khoảng 59.8 GB. Cell metadata dùng Parquet column projection nên không tải cột `audio`. `MAX_PARQUET_FILES=None` nghĩa là quét toàn bộ shard; đặt một số nguyên nhỏ để chạy thử nhanh.

In [ ]:
from __future__ import annotations

import io
import os
import re
import unicodedata
from collections import Counter
from itertools import islice
from pathlib import Path

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import seaborn as sns
import soundfile as sf
from IPython.display import Audio, Markdown, display
from huggingface_hub import HfFileSystem
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.unicode_minus"] = False

DATASET_ID = "nguyendv02/ViMD_Dataset"
RANDOM_SEED = 42
MAX_PARQUET_FILES = None  # Ví dụ: 12 để chạy thử nhanh; None để đọc toàn bộ metadata.
OUTPUT_DIR = Path("/content/vimd_eda")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LOAD_AUDIO_SAMPLES = False
AUDIO_SAMPLE_SIZE_PER_SPLIT = 2
AUDIO_PREVIEW_SECONDS = 12

print("Dataset :", DATASET_ID)
print("Output  :", OUTPUT_DIR.resolve())
print("Audio EDA:", "bật" if LOAD_AUDIO_SAMPLES else "tắt")

## 2. Liệt kê shard và đọc metadata

Dataset công khai thường không cần token. Nếu Hugging Face yêu cầu xác thực hoặc bị giới hạn truy cập, đặt token trong biến môi trường `HF_TOKEN`.

In [ ]:
hf_token = os.getenv("HF_TOKEN")
fs = HfFileSystem(token=hf_token)
repo_prefix = f"datasets/{DATASET_ID}"
all_parquet_paths = sorted(fs.glob(f"{repo_prefix}/data/*.parquet"))

if not all_parquet_paths:
    raise FileNotFoundError(
        f"Không tìm thấy shard Parquet trong {DATASET_ID}. "
        "Hãy kiểm tra mạng, tên dataset hoặc HF_TOKEN."
    )

def split_from_path(path: str) -> str:
    name = path.rsplit("/", 1)[-1]
    raw_split = name.split("-", 1)[0].lower()
    return {"validation": "valid", "val": "valid"}.get(raw_split, raw_split)

shard_inventory = pd.Series(
    [split_from_path(path) for path in all_parquet_paths], name="split"
).value_counts().sort_index().to_frame("parquet_shards")
display(shard_inventory)
print(f"Tổng cộng: {len(all_parquet_paths)} shard Parquet.")

parquet_paths = all_parquet_paths[:MAX_PARQUET_FILES] if MAX_PARQUET_FILES else all_parquet_paths
if len(parquet_paths) < len(all_parquet_paths):
    print(f"CHẾ ĐỘ NHANH: chỉ đọc {len(parquet_paths)}/{len(all_parquet_paths)} shard.")

In [ ]:
with fs.open(parquet_paths[0], "rb") as stream:
    parquet_schema = pq.ParquetFile(stream).schema_arrow

available_schema_columns = parquet_schema.names
print("Schema Parquet:", available_schema_columns)

requested_columns = [
    "region",
    "province_code",
    "province_name",
    "province",
    "filename",
    "text",
    "speakerID",
    "gender",
    "length",
]
metadata_columns = [column for column in requested_columns if column in available_schema_columns]
missing_optional_columns = [column for column in requested_columns if column not in available_schema_columns]

if not metadata_columns:
    raise ValueError("Không tìm thấy cột metadata mong đợi trong schema.")

print("Sẽ đọc        :", metadata_columns)
print("Không có/khác tên:", missing_optional_columns)

In [ ]:
metadata_frames = []

for path in tqdm(parquet_paths, desc="Đọc metadata Parquet"):
    with fs.open(path, "rb") as stream:
        frame = pd.read_parquet(stream, columns=metadata_columns)
    frame["split"] = split_from_path(path)
    frame["source_parquet"] = path.rsplit("/", 1)[-1]
    metadata_frames.append(frame)

df = pd.concat(metadata_frames, ignore_index=True)
df.insert(0, "row_id", np.arange(len(df), dtype=int))

print(f"Đã đọc {len(df):,} dòng × {df.shape[1]} cột, không tải cột audio.")
display(df.head())

## 3. Chuẩn hóa và làm giàu metadata

In [ ]:
def normalize_text(value: object) -> str:
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFC", str(value)).strip()
    return re.sub(r"\s+", " ", text)

def humanize_province(value: object) -> str:
    if pd.isna(value):
        return "Không rõ"
    return re.sub(r"(?<=[a-z])(?=[A-Z])", " ", str(value)).strip()

string_columns = ["region", "province_name", "province", "filename", "text", "speakerID", "split"]
for column in [column for column in string_columns if column in df.columns]:
    df[column] = df[column].astype("string").str.strip()

if "province_name" not in df.columns and "province" in df.columns:
    df["province_name"] = df["province"]
if "province_name" not in df.columns:
    df["province_name"] = "Không rõ"

df["province_label"] = df["province_name"].map(humanize_province)
df["text_clean"] = df.get("text", pd.Series(index=df.index, dtype="string")).map(normalize_text)
df["char_count"] = df["text_clean"].str.len()
df["word_count"] = df["text_clean"].str.split().str.len()
gender_numeric = pd.to_numeric(df.get("gender", pd.Series(index=df.index)), errors="coerce")
df["gender_label"] = gender_numeric.map({0: "female", 1: "male"}).fillna("unknown")
df["has_terminal_punctuation"] = df["text_clean"].str.contains(r"[.!?…][\"'’”)]?$", regex=True)
df["contains_digit"] = df["text_clean"].str.contains(r"\d", regex=True)
df["is_duplicate_filename"] = df.get("filename", pd.Series(index=df.index)).duplicated(keep=False)
df["is_duplicate_text"] = df["text_clean"].duplicated(keep=False)

if "length" in df.columns:
    df["length"] = pd.to_numeric(df["length"], errors="coerce")
    df["words_per_minute"] = np.where(
        df["length"].gt(0), 60 * df["word_count"] / df["length"], np.nan
    )
else:
    print("Schema không có cột length; thời lượng sẽ chỉ được đo cho audio mẫu ở phần tùy chọn.")

display(df.head())

## 4. Tổng quan và đối chiếu thống kê công bố

Dataset card công bố 18,949 bản ghi: train 15,023; valid 1,900; test 2,026. Bảng dưới đối chiếu số dòng đọc được với các con số đó. Nếu chạy `MAX_PARQUET_FILES`, chênh lệch là bình thường.

In [ ]:
published = pd.DataFrame(
    {
        "split": ["train", "valid", "test"],
        "published_records": [15023, 1900, 2026],
        "published_speakers": [10291, 1320, 1344],
        "published_hours": [81.43, 10.26, 10.87],
        "published_words": [981391, 125305, 132471],
    }
)
observed = (
    df.groupby("split")
    .agg(
        observed_records=("row_id", "size"),
        observed_speakers=("speakerID", "nunique"),
        observed_words=("word_count", "sum"),
    )
    .reset_index()
)
comparison = published.merge(observed, on="split", how="outer")
comparison["record_delta"] = comparison["observed_records"] - comparison["published_records"]
comparison["speaker_delta"] = comparison["observed_speakers"] - comparison["published_speakers"]
display(comparison)

summary = pd.Series(
    {
        "rows": len(df),
        "splits": df["split"].nunique(),
        "regions": df["region"].nunique() if "region" in df else np.nan,
        "provinces": df["province_name"].nunique(),
        "speakers": df["speakerID"].nunique() if "speakerID" in df else np.nan,
        "total_words_calculated": int(df["word_count"].sum()),
        "parquet_shards_read": len(parquet_paths),
    },
    name="value",
).to_frame()
display(summary)

In [ ]:
missing = (
    df.isna().sum()
    .to_frame("missing_count")
    .assign(missing_percent=lambda x: (100 * x["missing_count"] / len(df)).round(2))
    .sort_values("missing_count", ascending=False)
)
display(Markdown("### Giá trị thiếu"))
display(missing)

quality_checks = pd.Series(
    {
        "empty_text": int(df["text_clean"].eq("").sum()),
        "duplicate_filename_rows": int(df["is_duplicate_filename"].sum()),
        "duplicate_text_rows": int(df["is_duplicate_text"].sum()),
        "missing_terminal_punctuation": int((~df["has_terminal_punctuation"]).sum()),
        "text_containing_digits": int(df["contains_digit"].sum()),
        "speaker_in_multiple_regions": int((df.groupby("speakerID")["region"].nunique() > 1).sum()),
        "speaker_in_multiple_provinces": int((df.groupby("speakerID")["province_name"].nunique() > 1).sum()),
        "speaker_with_inconsistent_gender": int((df.groupby("speakerID")["gender_label"].nunique() > 1).sum()),
    },
    name="count",
).to_frame()
display(Markdown("### Kiểm tra chất lượng metadata"))
display(quality_checks)

## 5. Phân bố split, vùng, tỉnh và giới tính

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
for column, ax, color in zip(
    ["split", "region", "gender_label"],
    axes,
    ["#4C78A8", "#72B7B2", "#F58518"],
):
    counts = df[column].fillna("missing").value_counts().rename_axis(column).reset_index(name="records")
    sns.barplot(data=counts, x=column, y="records", color=color, ax=ax)
    ax.set_title(f"Phân bố {column}")
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
province_stats = (
    df.groupby(["region", "province_code", "province_label"], dropna=False)
    .agg(
        records=("row_id", "size"),
        speakers=("speakerID", "nunique"),
        words=("word_count", "sum"),
        female_records=("gender_label", lambda s: int((s == "female").sum())),
        male_records=("gender_label", lambda s: int((s == "male").sum())),
    )
    .reset_index()
    .sort_values("records", ascending=False)
)
display(province_stats)

plt.figure(figsize=(13, max(10, 0.27 * len(province_stats))))
sns.barplot(data=province_stats, y="province_label", x="records", hue="region", dodge=False)
plt.title("Số bản ghi theo tỉnh")
plt.xlabel("Số bản ghi")
plt.ylabel("Tỉnh")
plt.legend(title="Vùng")
plt.tight_layout()
plt.show()

In [ ]:
province_split = pd.crosstab(df["province_label"], df["split"])
province_split = province_split.loc[province_split.sum(axis=1).sort_values(ascending=False).index]

plt.figure(figsize=(10, max(10, 0.28 * len(province_split))))
sns.heatmap(province_split, cmap="Blues", annot=True, fmt="g", cbar=False)
plt.title("Độ phủ tỉnh trong từng split")
plt.xlabel("Split")
plt.ylabel("Tỉnh")
plt.tight_layout()
plt.show()

missing_split_coverage = (province_split == 0).stack()
missing_split_coverage = missing_split_coverage[missing_split_coverage].rename("missing").reset_index()
display(Markdown("### Tổ hợp tỉnh–split không có dữ liệu"))
display(missing_split_coverage if not missing_split_coverage.empty else pd.DataFrame({"Kết quả": ["Mọi tỉnh đều có mặt trong mọi split đã đọc"]}))

In [ ]:
region_gender_split = pd.crosstab(
    [df["region"], df["gender_label"]],
    df["split"],
)
display(region_gender_split)

imbalance = province_stats.groupby("region")["records"].agg(["min", "median", "max", "mean"])
imbalance["max_to_min_ratio"] = imbalance["max"] / imbalance["min"].replace(0, np.nan)
display(Markdown("### Mất cân bằng số bản ghi giữa các tỉnh trong từng vùng"))
display(imbalance.round(2))

## 6. Kiểm tra rò rỉ giữa các split

Speaker hoặc cùng một audio xuất hiện ở nhiều split có thể làm kết quả đánh giá quá lạc quan. Transcript trùng giữa split không nhất thiết là lỗi, nhưng cần được báo cáo tùy bài toán.

In [ ]:
speaker_split_counts = df.groupby("speakerID")["split"].nunique()
leaked_speakers = speaker_split_counts[speaker_split_counts > 1].index
speaker_leakage = (
    df[df["speakerID"].isin(leaked_speakers)]
    .groupby("speakerID")
    .agg(
        splits=("split", lambda s: " | ".join(sorted(s.unique()))),
        records=("row_id", "size"),
        regions=("region", lambda s: " | ".join(sorted(s.dropna().unique()))),
        provinces=("province_label", lambda s: " | ".join(sorted(s.dropna().unique()))),
    )
    .reset_index()
    .sort_values("records", ascending=False)
)
display(Markdown(f"### Speaker xuất hiện ở nhiều split: {len(speaker_leakage):,}"))
display(speaker_leakage.head(50))

In [ ]:
def cross_split_duplicates(column: str) -> pd.DataFrame:
    split_counts = df.groupby(column, dropna=False)["split"].nunique()
    duplicate_values = split_counts[split_counts > 1].index
    return (
        df[df[column].isin(duplicate_values)]
        .groupby(column, dropna=False)
        .agg(
            splits=("split", lambda s: " | ".join(sorted(s.unique()))),
            records=("row_id", "size"),
            speakers=("speakerID", "nunique"),
        )
        .reset_index()
        .sort_values("records", ascending=False)
    )

filename_leakage = cross_split_duplicates("filename")
text_leakage = cross_split_duplicates("text_clean")

display(Markdown(f"### Filename xuất hiện ở nhiều split: {len(filename_leakage):,}"))
display(filename_leakage.head(30))
display(Markdown(f"### Transcript chuẩn hóa xuất hiện ở nhiều split: {len(text_leakage):,}"))
display(text_leakage.head(30))

## 7. Phân tích transcript

In [ ]:
display(
    df[["char_count", "word_count"]]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
    .T
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(df["word_count"], bins=40, kde=True, ax=axes[0], color="#4C78A8")
axes[0].set(title="Phân bố số từ", xlabel="Số từ")
sns.boxplot(data=df, x="region", y="word_count", ax=axes[1], color="#72B7B2")
axes[1].set(title="Độ dài transcript theo vùng", xlabel="Vùng", ylabel="Số từ")
plt.tight_layout()
plt.show()

In [ ]:
VI_STOPWORDS = {
    "và", "là", "của", "có", "thì", "một", "những", "cái", "được",
    "cho", "trong", "để", "với", "cũng", "đã", "này", "đó", "ở",
    "mà", "các", "không", "tôi", "mình", "em", "chúng", "rất", "về",
}

token_counter = Counter()
for text in df["text_clean"]:
    tokens = re.findall(r"[^\W\d_]+", text.lower(), flags=re.UNICODE)
    token_counter.update(token for token in tokens if len(token) > 1 and token not in VI_STOPWORDS)

top_tokens = pd.DataFrame(token_counter.most_common(30), columns=["token", "count"])
display(top_tokens)

plt.figure(figsize=(10, 8))
sns.barplot(data=top_tokens, x="count", y="token", color="#59A14F")
plt.title("30 từ xuất hiện nhiều nhất sau khi bỏ stopword cơ bản")
plt.xlabel("Tần suất")
plt.ylabel("Từ")
plt.tight_layout()
plt.show()

## 8. Thời lượng (nếu metadata có cột `length`)

In [ ]:
if "length" in df.columns:
    display(df[["length", "words_per_minute"]].describe().T)
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    sns.histplot(df["length"].dropna(), bins=40, kde=True, ax=axes[0])
    axes[0].set(title="Phân bố thời lượng", xlabel="Giây")
    sns.histplot(df["words_per_minute"].replace([np.inf, -np.inf], np.nan).dropna(), bins=40, kde=True, ax=axes[1])
    axes[1].set(title="Tốc độ nói gần đúng", xlabel="Từ/phút")
    plt.tight_layout()
    plt.show()
else:
    display(Markdown(
        "Parquet không có cột `length`. Dataset Viewer tính thời lượng từ audio; "
        "notebook không quét toàn bộ audio chỉ để lấy chỉ số này nhằm tránh tải gần 60 GB."
    ))

## 9. Shortlist để nghe thủ công

Mỗi tổ hợp vùng–tỉnh–giới tính lấy tối đa một bản ghi, ưu tiên speaker có nhiều câu và transcript gần 30 từ. Shortlist không đồng nghĩa audio đã sạch hoặc được phép dùng để clone giọng.

In [ ]:
speaker_record_counts = df.groupby("speakerID").size().rename("speaker_records")
candidate_pool = df.join(speaker_record_counts, on="speakerID")
candidate_pool["target_word_distance"] = (candidate_pool["word_count"] - 30).abs()
candidate_pool["metadata_complete"] = candidate_pool[
    ["region", "province_name", "speakerID", "gender_label", "filename", "text_clean"]
].notna().all(axis=1)

shortlist = (
    candidate_pool
    .sort_values(
        ["metadata_complete", "speaker_records", "target_word_distance"],
        ascending=[False, False, True],
    )
    .groupby(["region", "province_name", "gender_label"], dropna=False, as_index=False)
    .head(1)
    [[
        "row_id", "split", "region", "province_code", "province_name",
        "gender_label", "speakerID", "speaker_records", "filename",
        "word_count", "text_clean",
    ]]
    .sort_values(["region", "province_code", "gender_label"])
    .reset_index(drop=True)
)
display(shortlist)

## 10. EDA audio có kiểm soát (tùy chọn)

Đổi `LOAD_AUDIO_SAMPLES=True` để lấy một số ít audio đầu mỗi split bằng streaming. Phần này đo sample rate, thời lượng, peak/RMS, clipping, tỷ lệ frame yên lặng; đồng thời cho nghe 12 giây đầu và xem waveform/spectrogram.

In [ ]:
def decode_audio_object(audio_obj: dict) -> tuple[np.ndarray, int]:
    payload = audio_obj.get("bytes")
    path = audio_obj.get("path")
    if payload is not None:
        signal, sample_rate = sf.read(io.BytesIO(payload), dtype="float32", always_2d=True)
    elif path:
        signal, sample_rate = sf.read(path, dtype="float32", always_2d=True)
    else:
        raise ValueError("Audio không có bytes hoặc path.")
    return signal.mean(axis=1), int(sample_rate)

def audio_metrics(signal: np.ndarray, sample_rate: int) -> dict:
    eps = np.finfo(np.float32).eps
    peak = float(np.max(np.abs(signal))) if signal.size else 0.0
    rms = float(np.sqrt(np.mean(np.square(signal)))) if signal.size else 0.0
    frame_rms = librosa.feature.rms(y=signal, frame_length=2048, hop_length=512)[0]
    frame_db = librosa.amplitude_to_db(np.maximum(frame_rms, eps), ref=1.0)
    return {
        "sample_rate": sample_rate,
        "decoded_duration_s": len(signal) / sample_rate,
        "peak_dbfs": 20 * np.log10(max(peak, eps)),
        "rms_dbfs": 20 * np.log10(max(rms, eps)),
        "clipping_percent": 100 * float(np.mean(np.abs(signal) >= 0.999)),
        "quiet_frame_percent": 100 * float(np.mean(frame_db < -40)),
    }

audio_examples = []
audio_quality = pd.DataFrame()

if LOAD_AUDIO_SAMPLES:
    from datasets import Audio as HFAudio
    from datasets import load_dataset

    metric_rows = []
    for split_name in sorted(df["split"].dropna().unique()):
        stream = load_dataset(DATASET_ID, split=split_name, streaming=True, token=hf_token)
        stream = stream.cast_column("audio", HFAudio(decode=False))
        for local_index, row in enumerate(islice(iter(stream), AUDIO_SAMPLE_SIZE_PER_SPLIT)):
            signal, sample_rate = decode_audio_object(row["audio"])
            metric_rows.append({
                "split": split_name,
                "sample_index": local_index,
                "speakerID": row.get("speakerID"),
                "region": row.get("region"),
                "province_name": row.get("province_name"),
                **audio_metrics(signal, sample_rate),
            })
            audio_examples.append({
                "split": split_name,
                "row": row,
                "signal": signal[: sample_rate * AUDIO_PREVIEW_SECONDS],
                "sample_rate": sample_rate,
            })
    audio_quality = pd.DataFrame(metric_rows)
    display(audio_quality)
else:
    print("Audio EDA đang tắt. Đổi LOAD_AUDIO_SAMPLES=True ở phần cấu hình rồi chạy lại cell này.")

In [ ]:
if audio_examples:
    for example in audio_examples:
        row = example["row"]
        signal = example["signal"]
        sample_rate = example["sample_rate"]
        title = f"{example['split']} · {row.get('speakerID')} · {row.get('region')} · {row.get('province_name')}"

        display(Markdown(f"### {title}"))
        display(Markdown(f"**Transcript:** {row.get('text', '')}"))
        display(Audio(signal, rate=sample_rate))

        fig, axes = plt.subplots(1, 2, figsize=(16, 4))
        librosa.display.waveshow(signal, sr=sample_rate, ax=axes[0])
        axes[0].set(title=f"Waveform — {AUDIO_PREVIEW_SECONDS} giây đầu", xlabel="Thời gian (s)")
        mel = librosa.feature.melspectrogram(y=signal, sr=sample_rate, n_mels=80, fmax=min(8000, sample_rate // 2))
        mel_db = librosa.power_to_db(mel, ref=np.max)
        image = librosa.display.specshow(mel_db, sr=sample_rate, x_axis="time", y_axis="mel", ax=axes[1])
        axes[1].set(title="Mel spectrogram")
        fig.colorbar(image, ax=axes[1], format="%+2.0f dB")
        plt.tight_layout()
        plt.show()
else:
    print("Chưa có audio mẫu để hiển thị.")

## 11. Xuất kết quả

In [ ]:
metadata_output = OUTPUT_DIR / "vimd_metadata_enriched.csv"
province_output = OUTPUT_DIR / "vimd_province_summary.csv"
shortlist_output = OUTPUT_DIR / "vimd_reference_shortlist.csv"
leakage_output = OUTPUT_DIR / "vimd_speaker_split_leakage.csv"

df.to_csv(metadata_output, index=False, encoding="utf-8-sig")
province_stats.to_csv(province_output, index=False, encoding="utf-8-sig")
shortlist.to_csv(shortlist_output, index=False, encoding="utf-8-sig")
speaker_leakage.to_csv(leakage_output, index=False, encoding="utf-8-sig")

print("Đã lưu:")
for output_path in [metadata_output, province_output, shortlist_output, leakage_output]:
    print(" -", output_path)

## Cách diễn giải kết quả

- Nếu `record_delta` khác 0 khi đã đọc toàn bộ shard, kiểm tra phiên bản repository và cách tác giả đếm bản ghi.
- Nếu có speaker leakage, nên chia lại dữ liệu theo `speakerID` trước khi huấn luyện/đánh giá.
- Nếu một số tỉnh hoặc giới tính thiếu ở validation/test, metric tổng có thể che giấu chất lượng thấp trên nhóm hiếm.
- Transcript trùng giữa split cần được đánh giá theo bài toán: nghiêm trọng hơn với ASR đọc câu cố định, ít nghiêm trọng hơn nếu speaker hoàn toàn tách biệt.
- Shortlist chỉ là điểm bắt đầu; luôn nghe thủ công và kiểm tra giấy phép/quyền sử dụng trước khi dùng audio cho TTS.